# Environment Setup

In [1]:
import sys 
from pathlib import Path
import os 
import requests

print(f"Python Version: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
print(f"Environment: {sys.executable}")

# Find project root and add to Python path
current_dir = Path.cwd()
if current_dir.name == "week3" and current_dir.parent.name == "notebooks":
    project_root = current_dir.parent.parent
elif (current_dir / "compose.yml").exists():
    project_root = current_dir
else:
    project_root = None

if project_root and (project_root / "compose.yml").exists():
    print(f"Project root: {project_root}")
    sys.path.insert(0, str(project_root))
else:
    print("Missing compose.yml - check directory")
    exit()

Python Version: 3.12.12
Environment: /Users/anhvietpham/Documents/AI/Project-practice/chatbot/.venv/bin/python
Project root: /Users/anhvietpham/Documents/AI/Project-practice/chatbot


# 1. Infrashtructure Verification

In [2]:
print("WEEK 3 PREREQUISITE CHECK")
print("=" * 50)

services_to_test = {
    "FastAPI": "http://localhost:8001/api/v1/health",
    "PostgreSQL (via API)": "http://localhost:8001/api/v1/health", 
    "OpenSearch": "http://localhost:9200/_cluster/health",
    "Airflow": "http://localhost:8080/health"  
}

all_healthy = True

for service_name, url in services_to_test.items(): 
    try: 
        response = requests.get(url, timeout=5)
        if response.status_code == 200: 
            print(f"✓ {service_name}: Healthy")
        else: 
            print(f"✗ {service_name}: HTTP {response.status_code}")
            all_healthy = False
    except requests.exceptions.ConnectionError:
        print(f"✗ {service_name}: Not accessible")
        all_healthy = False
    except Exception as e:
        print(f"✗ {service_name}: {type(e).__name__}")
        all_healthy = False

print()
if all_healthy:
    print("All services healthy! Ready for Week 3 OpenSearch integration.")
else:
    print("Some services need attention. Please run: docker compose up --build")

WEEK 3 PREREQUISITE CHECK
✓ FastAPI: Healthy
✓ PostgreSQL (via API): Healthy
✓ OpenSearch: Healthy
✗ Airflow: HTTP 404

Some services need attention. Please run: docker compose up --build


# 2. OpenSearch Client Setup

In [3]:
from src.services.opensearch.factory import make_opensearch_client
from opensearchpy import OpenSearch

print("OpenSearch Client Setup...")
print("=" * 50)

# Create OpenSearch client using factory pattern 
opensearch_client = make_opensearch_client()

# Override for notebook execution (localhost instead of container hostname)
opensearch_client.host = "http://localhost:9200"
opensearch_client.client = OpenSearch(
    hosts = ["http://localhost:9200"],
    http_compress = True, 
    use_ssl = False, 
    verify_certs = False, 
    ssl_assert_hostname = False, 
    ssl_show_warn = False,
)

print(f"Client configured with host: {opensearch_client.host}")
print(f"Index name: {opensearch_client.index_name}")

is_healthy = opensearch_client.health_check()
if is_healthy:
    print("✓ OpenSearch health check: PASSED")
else:
    print("✗ OpenSearch health check: FAILED") 


OpenSearch Client Setup...
Client configured with host: http://localhost:9200
Index name: chatbot-papers-chunks_v2
✓ OpenSearch health check: PASSED


# Create Index

In [4]:
print("INDEX CREATION")
print("=" * 50)

try: 
    index_exists = opensearch_client.client.indices.exists(index=opensearch_client.index_name)

    if index_exists:
        print(f"Index '{opensearch_client.index_name}' already exists.")

        stats = opensearch_client.get_index_stats()
        if stats and 'error' not in stats:
            print(f"\nCurrent Statistics:")
            print(f"   Documents: {stats.get('document_count', 0)}")
            print(f"   Size: {stats.get('size_in_bytes', 0):,} bytes")
    else: 
        print(f"Creating new index: {opensearch_client.index_name}")

        # Create the index with our custom mapping
        success = opensearch_client.create_index()

        if success:
            print(f"✓ Index created successfully!")
        else:
            print(f"✗ Index creation failed")
except Exception as e:
    print(f"✗ Error with index management: {e}")



INDEX CREATION
Index 'chatbot-papers-chunks_v2' already exists.

Current Statistics:
   Documents: 18
   Size: 391,367 bytes


## 3. Data Pipeline - Run Airflow DAG

The **arxiv_paper_ingestion** DAG automatically:
1. Fetches papers from arXiv API
2. Stores papers in PostgreSQL
3. **Indexes papers into OpenSearch**

### Instructions:

**Before proceeding, run the Airflow DAG:**

1. Open Airflow UI: http://localhost:8080
2. Login: username `admin`, password `admin`
3. Find **`arxiv_paper_ingestion`** DAG
4. Click the DAG name to open it
5. Click **"Trigger DAG"** button (▶️ play icon)
6. Wait ~10 minutes for completion
7. Check that all tasks turn green

Then run the cell below to verify:


In [5]:
# Verify Data Pipeline Results
print("VERIFYING DATA PIPELINE")
print("=" * 50)

stats = opensearch_client.get_index_stats()

if stats and 'error' not in stats:
    doc_count = stats.get('document_count', 0)

    if doc_count > 0: 
        print(f" Success! Found {doc_count} documents in OpenSearch")

        # Show sample papers
        sample = opensearch_client.search_papers("", size=3, latest=False)
        if sample: 
            print(f"\nSample Papers:")
            for i, paper in enumerate(sample['hits'], 1): 
                title = paper.get('title', 'Unknown')[:60]
                print(f" {i}. {title}")
    else: 
        print("No documents in OpenSearch yet!")
        print("\nPlease run the Airflow DAG first (see instructions above)")
else: 
    print("Could not retrieve index stats")

VERIFYING DATA PIPELINE
 Success! Found 18 documents in OpenSearch

Sample Papers:
 1. Hierarchical Cooperative Multi-Agent Reinforcement Learning 
 2. Hierarchical Cooperative Multi-Agent Reinforcement Learning 
 3. Hierarchical Cooperative Multi-Agent Reinforcement Learning 


In [6]:
print(opensearch_client.index_name)

chatbot-papers-chunks_v2


In [7]:
opensearch_client.client.indices.get_mapping(index=opensearch_client.index_name)

{'chatbot-papers-chunks_v2': {'mappings': {'dynamic': 'strict',
   'properties': {'abstract': {'type': 'text', 'analyzer': 'text_analyzer'},
    'arxiv_id': {'type': 'keyword'},
    'authors': {'type': 'text',
     'fields': {'keyword': {'type': 'keyword', 'ignore_above': 512}},
     'analyzer': 'text_analyzer'},
    'categories': {'type': 'keyword'},
    'chunk_id': {'type': 'keyword'},
    'chunk_index': {'type': 'integer'},
    'chunk_text': {'type': 'text', 'analyzer': 'text_analyzer'},
    'chunk_word_count': {'type': 'integer'},
    'created_at': {'type': 'date'},
    'embedding': {'type': 'knn_vector',
     'dimension': 1024,
     'method': {'engine': 'nmslib',
      'space_type': 'cosinesimil',
      'name': 'hnsw',
      'parameters': {'ef_construction': 512, 'm': 16}}},
    'embedding_model': {'type': 'keyword'},
    'end_char': {'type': 'integer'},
    'paper_id': {'type': 'keyword'},
    'published_date': {'type': 'date'},
    'section_title': {'type': 'keyword'},
    'star

# Simple BM25 Search

In [8]:
# Simple BM25 Search
print("Simple BM25 SEARCH")
print("=" * 40)

# Change this to any word from your papers
search_term = "learning"

print(f"Searching for: {search_term}")

results = opensearch_client.search_papers(
    query=search_term,
    size=5
)

if results.get('hits'):
    print(f"Found {results.get('total', 0)} total matches\n")
    
    for i, paper in enumerate(results['hits'], 1):
        print(f"{i}. {paper.get('title', 'Unknown')[:70]}...")
        print(f"   Score: {paper.get('score', 0):.2f}")
        print(f"   arXiv ID: {paper.get('arxiv_id', 'N/A')}\n")
else:
    print("No results found. Try searching for:")
    print("  • 'neural', 'model', 'algorithm'")
    print("  • Use '*' to see all papers")

Simple BM25 SEARCH
Searching for: learning
Found 18 total matches

1. Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
   Score: 0.52
   arXiv ID: 1912.03558v3

2. Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
   Score: 0.51
   arXiv ID: 1912.03558v3

3. Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
   Score: 0.50
   arXiv ID: 1912.03558v3

4. Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
   Score: 0.49
   arXiv ID: 1912.03558v3

5. Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
   Score: 0.48
   arXiv ID: 1912.03558v3



## 5. Advanced OpenSearch Queries

5.1 Match Query

In [9]:
print("MATCH QUERY - Single Field Search")
print("=" * 40)

query = {
    "query": {
        "match": {
            "title": "Multi-Agent"
        }
    }, 
    "size": 3
}

response = opensearch_client.client.search(
    index=opensearch_client.index_name,
    body=query
)


print(f"Found {response['hits']['total']['value']} results\n")

for hit in response['hits']['hits']: 
    print(f"Title: {hit['_source']['title'][:70]}...")


MATCH QUERY - Single Field Search
Found 18 results

Title: Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
Title: Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
Title: Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...


5.2 Multi-Match Query

In [10]:
print("MULTI-MATCH QUERY - Search Multiple Fields")
print("=" * 40)

query = {
    "query": {
        "multi_match": {
            "query": "Multi-Agent",
            "fields": ["title^2", "abstract", "authors"], 
            "type": "best_fields",
        }
    }, 
    "size": 3
}

response = opensearch_client.client.search(
    index=opensearch_client.index_name,
    body=query
)

print(f"Found {response['hits']['total']['value']} results\n")

for hit in response['hits']['hits']: 
    print(f"Title: {hit['_source']['title'][:70]}...")
    print(f"Score: {hit['_score']:.2f}")
    print(f"Authors: {', '.join(hit['_source']['authors'])}")

MULTI-MATCH QUERY - Search Multiple Fields
Found 18 results

Title: Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
Score: 0.11
Authors: U, n, k, n, o, w, n
Title: Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
Score: 0.11
Authors: U, n, k, n, o, w, n
Title: Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
Score: 0.11
Authors: U, n, k, n, o, w, n


5.3 Boosting Query

In [11]:
print("BOOSTING QUERY - Promote/Demote Results")
print("=" * 40)

query = {
    "query": {
        "boosting": {
            "positive": {
                "match": {
                    "abstract": "multi agent"
                }
            },
            "negative": {
                "match": {
                    "abstract": "multimodel"
                }
            },
            "negative_boost": 0.1
        }
    }, 
    "size": 3
}

response = opensearch_client.client.search(
    index=opensearch_client.index_name,
    body=query
)

print(f"Query: Boost 'multi agent', demote 'survey' papers\n")
print(f"Found {response['hits']['total']['value']} results\n")

for hit in response['hits']['hits']: 
    title = hit['_source']['title'][:70]
    abstract_snippet = hit['_source']['abstract'][:100]
    print(f"Title: {title}...")
    print(f"Score: {hit['_score']:.2f}")
    print(f"Abstract: {abstract_snippet}...\n")

BOOSTING QUERY - Promote/Demote Results
Query: Boost 'multi agent', demote 'survey' papers

Found 18 results

Title: Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
Score: 0.08
Abstract: Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill Discovery

Jiachen Yang ∗ Geo...

Title: Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
Score: 0.08
Abstract: Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill Discovery

Jiachen Yang ∗ Geo...

Title: Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
Score: 0.08
Abstract: Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill Discovery

Jiachen Yang ∗ Geo...



5.4 Filter Query

In [12]:
print("FILTER QUERY - Category Filtering")
print("=" * 40)

query = {
    "query": {
        "bool": {
            "must": [
                {
                    "match": {
                        "abstract": "Hierarchical"
                    }
                }
            ],
            "filter": [
                {
                    "terms": {
                        "categories.keyword": ["cs.AI"]
                    }
                }
            ]
        }
    },
    "size": 3
}

response = opensearch_client.client.search(
    index=opensearch_client.index_name,
    body=query
)

print(f"Found {response['hits']['total']['value']} results\n")

for hit in response['hits']['hits']:
    title = hit['_source']['title'][:70]
    categories = ', '.join(hit['_source']['categories'])
    print(f"Title: {title}...")
    print(f"Categories: {categories}")
    print(f"Score: {hit['_score']:.2f}\n")

FILTER QUERY - Category Filtering
Found 0 results



In [13]:
q = {
    "query": {
        "bool": {
            "filter": [
                {"terms": {"categories.keyword": ["cs.AI"]}}
            ]
        }
    },
    "size": 5
}

resp = opensearch_client.client.search(
    index="chatbot-papers-chunks",
    body=q
)
print("Found", resp["hits"]["total"]["value"], "results")
for h in resp["hits"]["hits"]:
    print(h["_source"]["arxiv_id"], h["_source"]["categories"])

Found 18 results
1912.03558v3 ['cs.AI']
1912.03558v3 ['cs.AI']
1912.03558v3 ['cs.AI']
1912.03558v3 ['cs.AI']
1912.03558v3 ['cs.AI']


5.5 Sorting Query

In [14]:
print("SORTING QUERY - Latest Papers First")
print("=" * 40)

query = {
    "query": {
        "match_all": {}
    },
    "sort": [
        {
            "published_date": {
                "order": "desc"
            }
        }
    ],
    "size": 5
}

response = opensearch_client.client.search(
    index=opensearch_client.index_name, 
    body=query
)

print(f"Query: All papers sorted by publication date (newest first)\n")

for hit in response['hits']['hits']:
    title = hit['_source']['title'][:70]
    pub_date = hit['_source']['published_date'][:10]
    print(f"Date: {pub_date} | {title}...") 

SORTING QUERY - Latest Papers First
Query: All papers sorted by publication date (newest first)

Date: 2026-03-14 | Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
Date: 2026-03-14 | Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
Date: 2026-03-14 | Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
Date: 2026-03-14 | Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...
Date: 2026-03-14 | Hierarchical Cooperative Multi-Agent Reinforcement Learning with Skill...


5.6 Combined Query

In [15]:
print("COMBINED QUERY - Complex Search")
print("=" * 40)

query = {
    "query": {
        "bool": {
            "must": [
                {
                    "multi_match": {
                        "query": "transformer",
                        "fields": ["title^3", "abstract"],
                        "type": "best_fields"
                    }
                }
            ],
            "filter": [
                {
                    "range": {
                        "published_date": {
                            "gte": "2024-01-01"
                        }
                    }
                }
            ],
            "should": [
                {
                    "match": {
                        "categories": "cs.AI"
                    }
                }
            ]
        }
    },
    "sort": [
        "_score",
        {"published_date": {"order": "desc"}}
    ],
    "size": 3
}

response = opensearch_client.client.search(
    index=opensearch_client.index_name,
    body=query
)

print(f"Complex Query:")
print(f"  • Must contain 'transformer' (title boosted 3x)")
print(f"  • Filter: published after 2024-01-01")
print(f"  • Prefer: cs.AI category")
print(f"  • Sort: by relevance, then date\n")

print(f"Found {response['hits']['total']['value']} results\n")

for hit in response['hits']['hits']:
    title = hit['_source']['title'][:70]
    pub_date = hit['_source']['published_date'][:10]
    score = hit['_score']
    categories = ', '.join(hit['_source']['categories'][:2])
    
    print(f"Title: {title}...")
    print(f"  Date: {pub_date} | Score: {score:.2f}")
    print(f"  Categories: {categories}\n")

COMBINED QUERY - Complex Search
Complex Query:
  • Must contain 'transformer' (title boosted 3x)
  • Filter: published after 2024-01-01
  • Prefer: cs.AI category
  • Sort: by relevance, then date

Found 0 results

